In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import art
from art.local import LocalBackend
import asyncio
from dotenv import load_dotenv
import json
from openai.types.chat.chat_completion import ChatCompletion
import os

load_dotenv()

MODEL_NAME = "001"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

model = art.TrainableModel(
    name=MODEL_NAME, project="rock-paper-tool-use", base_model=BASE_MODEL
)
await model.register(LocalBackend())
client = model.openai_client()


def get_tool_call_id_and_move(chat_completion: ChatCompletion) -> tuple[str, str]:
    tool_calls = chat_completion.choices[0].message.tool_calls
    if not tool_calls:
        return "n/a", "nothing"
    tool_call = tool_calls[0]
    try:
        return tool_call.id, json.loads(tool_call.function.arguments)["move"]
    except json.JSONDecodeError:
        return tool_call.id, "nothing"
    except KeyError:
        return tool_call.id, "nothing"


async def rollout() -> art.Trajectory:
    tools: art.Tools = [
        {
            "type": "function",
            "function": {
                "name": "play_move",
                "description": "Play a move in rock-paper-scissors",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "move": {
                            "type": "string",
                            "enum": ["rock", "paper", "scissors"],
                            "description": "The move to play",
                        }
                    },
                    "required": ["move"],
                },
            },
        }
    ]
    trajectories = [
        art.Trajectory(
            messages_and_choices=[
                {
                    "role": "system",
                    "content": "You are a rock-paper-scissors playing agent. Use the play_move function tool to declare your moves.",
                },
                {
                    "role": "user",
                    "content": "What will your first move be?",
                },
            ],
            tools=tools,
            reward=0,
            metrics={
                "num_rounds": 0,
                "rock": 0,
                "paper": 0,
                "scissors": 0,
                "nothing": 0,
            },
        )
        for _ in range(2)
    ]
    for _ in range(10):
        chat_completions = await asyncio.gather(
            *[
                client.chat.completions.create(
                    messages=trajectory.messages(),
                    model=model,
                    tools=tools,
                    max_completion_tokens=100,
                )
                for trajectory, model in zip(trajectories, (MODEL_NAME, BASE_MODEL))
            ]
        )
        for trajectory, chat_completion in zip(trajectories, chat_completions):
            trajectory.messages_and_choices.append(chat_completion.choices[0])
        (id0, move0), (id1, move1) = list(
            map(get_tool_call_id_and_move, chat_completions)
        )
        beats = {
            "rock": "scissors",
            "paper": "rock",
            "scissors": "paper",
            "nothing": None,
        }
        if beats[move0] == move1:
            trajectories[0].reward += 1
        elif beats[move1] == move0:
            trajectories[1].reward += 1
        for trajectory in trajectories:
            trajectory.metrics["num_rounds"] += 1
        trajectories[0].metrics[move0] += 1
        trajectories[1].metrics[move1] += 1
        if max(t.reward for t in trajectories) > 2:
            break
        trajectories[0].messages_and_choices.extend(
            (
                {
                    "role": "tool",
                    "tool_call_id": id0,
                    "content": f"The other player played {move1}.",
                },
                {
                    "role": "user",
                    "content": "What will your next move be?",
                },
            )
        )
        trajectories[1].messages_and_choices.extend(
            (
                {
                    "role": "tool",
                    "tool_call_id": id1,
                    "content": f"The other player played {move0}.",
                },
                {
                    "role": "user",
                    "content": "What will your next move be?",
                },
            )
        )
    return trajectories[0]


for i in range(await model.get_step(), 100):
    trajectories = await art.gather_trajectories(
        (rollout() for _ in range(64)), max_exceptions=64
    )
    # if os.path.exists("debug.txt"):
    if len(trajectories) > 0:
        with open("debug.txt", "a") as f:
            f.write(f"\n################## Step {i} ##################\n")
            # f.write(f"Step {i}:\n")
            for trajectory in trajectories:
                f.writelines(f"{str(trajectory.messages_and_choices).split(',')}\n")


    await model.train(
        [art.TrajectoryGroup(trajectories)],
        config=art.TrainConfig(learning_rate=5e-5),
    )

/home/guycoh/AgentDaC/.venv/lib/python3.11/site-packages/art/__init__.py:11: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth  # type: ignore


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 05-26 14:15:42 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 05-26 14:15:42 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.5.1: Fast Qwen2 patching. Transformers: 4.51.3. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 47.334 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit with actual GPU utilization = 78.5%
Unsloth: Your GPU has CUDA compute capability 8.6 with VRAM = 47.33 GB.
Unsloth: Using conservativeness = 1.0. Chunke

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.80it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.80it/s]

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.57it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.57it/s]



INFO 05-26 14:16:08 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 05-26 14:16:08 [model_runner.py:1140] Model loading took 2.2564 GiB and 3.051839 seconds
INFO 05-26 14:16:18 [worker.py:287] Memory profiling takes 9.46 seconds
INFO 05-26 14:16:18 [worker.py:287] the current vLLM instance can use total_gpu_memory (47.33GiB) x gpu_memory_utilization (0.78) = 37.16GiB
INFO 05-26 14:16:18 [worker.py:287] model weights take 2.26GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 2.69GiB; the rest of the memory reserved for KV Cache is 32.15GiB.
INFO 05-26 14:16:18 [executor_base.py:112] # cuda blocks: 58526, # CPU blocks: 10922
INFO 05-26 14:16:18 [executor_base.py:117] Maximum concurrency for 32768 tokens per request: 28.58x
INFO 05-26 14:16:21 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.

Capturing CUDA graph shapes: 100%|██████████| 43/43 [00:38<00:00,  1.11it/s]


INFO 05-26 14:17:00 [model_runner.py:1592] Graph capturing finished in 39 secs, took 5.64 GiB
INFO 05-26 14:17:00 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 52.05 seconds
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm', 'q_norm', 'k_norm']
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm', 'q_norm', 'k_norm']


Unsloth 2025.5.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


gather:   0%|          | 0/64 [00:00<?, ?it/s]

AttributeError: 'KeyError' object has no attribute 'messages_and_choices'